# Nova Separação dos Dados


## Import Original

In [1]:
import pandas as pd

In [2]:
#Importar conjuntos de dados

df_dengue = pd.read_csv('database/Dengue_Tratado_Analise.csv')
df_malaria = pd.read_csv('database/Malaria_Tratado_Analise.csv')

print("Dengue: ")
print(df_dengue.columns)
print(f"Qtd Colunas: {len(df_dengue.columns)}")
print(df_dengue['Result'].value_counts())

print("\nMalária: ")
print(df_malaria.columns)
print(f"Qtd Colunas: {len(df_malaria.columns)}")
print(df_malaria['Result'].value_counts())

Dengue: 
Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Monocytes', 'Eosinophils', 'Total Cir.Eosinophils',
       'RBC COUNT', 'HTC/PCV(%)', 'MCV(fl)', 'MCH(pg)', 'MCHC(g/dl)',
       'RDW-CV(%)', 'Platelet Count', 'MPV(fl)', 'PDW(%)', 'PCT(%)', 'Result'],
      dtype='object')
Qtd Colunas: 18
Result
0    1170
1     913
Name: count, dtype: int64

Malária: 
Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
       'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result'],
      dtype='object')
Qtd Colunas: 11
Result
0    1120
1    1068
Name: count, dtype: int64


In [3]:
# Filtrar colunas da Dengue
df_dengue_filtrado = df_dengue[['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
 'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
 'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result']].copy()

In [4]:
print("Antes da remoção:")
print(f" Dengue: {df_dengue_filtrado.shape}")
print(f" Malaria: {df_malaria.shape}\n")

print("Proporções:")
print(df_dengue_filtrado['Result'].value_counts())
print(df_malaria['Result'].value_counts())

contagens_antes_dengue = df_dengue_filtrado['Result'].value_counts()
contagens_antes_malaria = df_malaria['Result'].value_counts()

common_features = list(df_dengue_filtrado.columns)

df_dengue_features = df_dengue_filtrado.reset_index()
df_malaria_features = df_malaria.reset_index()

# Encontra os pacientes em comum
common_records_merged = pd.merge(
    df_dengue_features,
    df_malaria_features,
    on=common_features,
    how='inner',
    suffixes=('_dengue', '_malaria')
)

# Índices a remover em cada df
indices_para_remover_dengue = common_records_merged['index_dengue'].unique()
indices_para_remover_malaria = common_records_merged['index_malaria'].unique()

# Cria um novo df apenas com os pacientes em comum (usando as features + Result)
df_pacientes_comuns = df_dengue_features.loc[
    df_dengue_features['index'].isin(indices_para_remover_dengue),
    common_features
].reset_index(drop=True)

# Remove os pacientes em comum de AMBOS os dfs originais
df_dengue_filtrado = df_dengue_filtrado.drop(index=indices_para_remover_dengue)
df_malaria = df_malaria.drop(index=indices_para_remover_malaria)

print("\nDepois da remoção:")
print(f" Df Dengue: {df_dengue_filtrado.shape}")
print(f" Df Malaria: {df_malaria.shape}")
print(f" Df Pacientes Comuns (removidos): {df_pacientes_comuns.shape}\n")

print("Value Counts Dengue:")
print(df_dengue_filtrado['Result'].value_counts())
print("\nValue Counts Malaria:")
print(df_malaria['Result'].value_counts())
print("\nValue Counts Pacientes Comuns:")
print(df_pacientes_comuns['Result'].value_counts())

# Contar quantos foram removidos de cada dataset
contagens_depois_dengue = df_dengue_filtrado['Result'].value_counts()
contagens_depois_malaria = df_malaria['Result'].value_counts()

print(f"\nDengue - Classe 0 removidos: {contagens_antes_dengue.get(0,0) - contagens_depois_dengue.get(0,0)}")
print(f"Dengue - Classe 1 removidos: {contagens_antes_dengue.get(1,0) - contagens_depois_dengue.get(1,0)}")
print(f"Malaria - Classe 0 removidos: {contagens_antes_malaria.get(0,0) - contagens_depois_malaria.get(0,0)}")
print(f"Malaria - Classe 1 removidos: {contagens_antes_malaria.get(1,0) - contagens_depois_malaria.get(1,0)}")

Antes da remoção:
 Dengue: (2083, 11)
 Malaria: (2188, 11)

Proporções:
Result
0    1170
1     913
Name: count, dtype: int64
Result
0    1120
1    1068
Name: count, dtype: int64

Depois da remoção:
 Df Dengue: (1599, 11)
 Df Malaria: (1704, 11)
 Df Pacientes Comuns (removidos): (484, 11)

Value Counts Dengue:
Result
0    928
1    671
Name: count, dtype: int64

Value Counts Malaria:
Result
0    878
1    826
Name: count, dtype: int64

Value Counts Pacientes Comuns:
Result
1    242
0    242
Name: count, dtype: int64

Dengue - Classe 0 removidos: 242
Dengue - Classe 1 removidos: 242
Malaria - Classe 0 removidos: 242
Malaria - Classe 1 removidos: 242


## Separação Teste/Treino $(60/40)$

In [5]:
# Treino e teste
X_dengue = df_dengue_filtrado.drop('Result', axis=1)
y_dengue = df_dengue_filtrado['Result']

In [6]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X_dengue, y_dengue, test_size=0.4, random_state=14, stratify=y_dengue)

In [7]:
dengue_train = pd.concat([x_train, y_train], axis=1)
dengue_train['tipo'] = 'treino'

dengue_test = pd.concat([x_test, y_test], axis=1)
dengue_test['tipo'] = 'teste'

dengue_df_final = pd.concat([dengue_train, dengue_test], axis=0)

In [8]:
dengue_df_final.columns

Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
       'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result', 'tipo'],
      dtype='object')

In [9]:
# Treino e teste
X_malaria = df_malaria.drop('Result', axis=1)
y_malaria = df_malaria['Result']

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X_malaria, y_malaria, test_size=0.4, random_state=14, stratify=y_malaria)

In [11]:
malaria_train = pd.concat([x_train, y_train], axis=1)
malaria_train['tipo'] = 'treino'

malaria_test = pd.concat([x_test, y_test], axis=1)
malaria_test['tipo'] = 'teste'

malaria_df = pd.concat([malaria_train, malaria_test], axis=0)

# Salvar

In [12]:
print(malaria_df.columns)
print(dengue_df_final.columns)
print(df_pacientes_comuns.columns)

Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
       'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result', 'tipo'],
      dtype='object')
Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
       'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result', 'tipo'],
      dtype='object')
Index(['Hemoglobin(Hb%)', 'Total WBC count(/cumm)', 'Neutrophils',
       'Lymphocytes', 'Total Cir.Eosinophils', 'HTC/PCV(%)', 'MCH(pg)',
       'MCHC(g/dl)', 'RDW-CV(%)', 'Platelet Count', 'Result'],
      dtype='object')


In [13]:
dengue_df_final.to_csv("database/Dengue.csv", index=False)
malaria_df.to_csv("database/Malaria.csv", index=False)
df_pacientes_comuns.to_csv("database/Shared.csv", index=False)